# Marseille et ses transports en commun : Construction base de données

---
## 1. Import
---

In [1]:
# Import modules

import folium 
import pandas as pd 
import geopandas as gpd
from pyproj import Transformer
import re
from shapely.geometry import Polygon, LineString, Point, MultiPoint
import branca.colormap as cm

# Import données
df_carreaux = gpd.read_file("../data/raw/insee/carreaux_200m_met.csv")
gdf_ligne = gpd.read_file("../data/raw/transport/lignes_tec_marseille.geojson")
gdf_arret = gpd.read_file("../data/raw/transport/arrets_tec_marseille.geojson")
gdf_contour = gpd.read_file("../data/raw/administratif/contours_geographiques.geojson")

# Création copie des fichiers
df_carreaux_clean = df_carreaux.copy()
gdf_ligne_clean = gdf_ligne.copy()
gdf_arret_clean = gdf_arret.copy()
gdf_contour_clean = gdf_contour.copy()

---
## 2. Paramètres
---

In [2]:
# Liste des codes géographiques pour la commune de Marseille
code_geo_marseille = [str(13200 + i) for i in range(1,17)]

# Coordonnées GPS Vieux Port
vieux_port = [43.296191, 5.371370]

# Extension de la surface du territoire étudié afin d'inclure les arrêts limitrophes
extension = 250

---
## 3. Fonctions
---

In [3]:
'''
Les données géographiques des carreaux sont insérées dans le code INSPIRE des données carroyées. Ce dernier contient les coordonnées du point
inférieur gauche du carreau. Le but de cette fonction est d'obtenir les coordonnées exprimées selon le CRS EPSG:4326 (afin de normaliser les données
géographiques sur l'ensemble des données utilisées) des quatres sommets des carreaux.
'''

def code_to_square_coord(inspire_id):
    
    list_id = []
    list_coord_bon = []

    transformer = Transformer.from_crs("EPSG:3035", "EPSG:4326", always_xy=True)
    
    for i in inspire_id :
        # Extraire les coordonnées du coin inférieur gauche (codé 0,0) avec une regex
        match = re.search(r'N(\d+)E(\d+)', i)
        if not match:
            raise ValueError("Identifiant INSPIRE invalide")
    
        y_0 = int(match.group(1))  # Nord
        x_0 = int(match.group(2))  # Est
    
        # Générer les coordonnées nécessaire à la formation du carreau
        '''
        Selon la définition du sytème de coordonnées EPSG:3035, les coordonnées sont exprimées en mètre. Ainsi, pour se déplacer de 200m vers l'Est, 
        soit la longueur du côté du carreau, il faut ajouter 200 à la coordonnée x_0. On obtiendra la coordonnée x_1. De la même façon, pour se déplacer vers le Nord, il faut ajouter
        200 à la coordonnée y_0. On obtiendra la coordonnée y_1.
        '''
    
        y_1 = y_0 + 200
        x_1 = x_0 + 200  

        carreau = [(x_0, y_0), (x_1, y_0), (x_1, y_1), (x_0, y_1)]
    
        # Transformer de EPSG:3035 → WGS84 (EPSG:4326)
        coord = []
        for pt in transformer.itransform(carreau) :
            coord.append(pt)

        # Ajout de l'id et des coordonnées à la liste
        list_id.append(i)
        list_coord_bon.append(Polygon(coord))
    
    # Création du GeoDataFrame : 
    geodf = gpd.GeoDataFrame(
        {"idcar_200m" : list_id,
         "coord_carreau" : list_coord_bon}
    )

    # Définition de la colonne geometry active
    geodf = geodf.set_geometry("coord_carreau")

    # Définition du CRS 
    geodf = geodf.set_crs("EPSG:4326")

    return geodf

In [4]:
'''
L'objectif est de sélectionner les coordonnées géographiques du territoire marseillais
'''

def select_marseille(gdf) :
    
    # Sélection des arrondissements de la ville de Marseille
    gdf = gdf[gdf["com_arm_current_code"].str.contains("|".join(code_geo_marseille))]

    return gdf

In [5]:
'''
Cette fonction a pour objectif de créer un polygone intégrant à la fois l'intégralité du territoire géographique de la ville de Marseille ainsi que 
l'intégralité du territoire étudié à travers les données carroyées de l'INSEE, en sachant que ces deux zones ne se recouvrent pas parfaitement.
'''

def tracer_territoire_etudie(gdf_marseille, gdf_carreaux) :

    # Récupération des polygones des arrondissements de Marseille et des carreaux de l'INSEE
    polygons = list(gdf_marseille.geometry) + list(gdf_carreaux.coord_carreau)

    # Création GeoDataFrame avec les Polygones des arrondissements et les Polygones des carreaux étudiés
    gdf_territoire_etudie = gpd.GeoDataFrame({"geometry" : polygons})
    gdf_territoire_etudie = gdf_territoire_etudie.set_geometry("geometry")

    # Fusion des Polygones : 
    territoire_etudie = gdf_territoire_etudie.union_all()

    return territoire_etudie

In [6]:
'''
Cette fonction permet de sélectionner les arrêts de la base de données situés dans la zone définie par le territoire de la ville de Marseille ou dans
un carreau au moins en partie sur le territoire marseillais ou dans un rayon donné de ces zones. Le rayon est déterminé dans les paramètres sous le 
de "tolerance".
'''

def select_stop(gdfstop, gdfcarreaux, gdfmarseille, tolerance) : 

    polygons = []
    
    gdfc = gdfcarreaux.copy()
    gdfm = gdfmarseille.copy()
    gdfs = gdfstop.copy()

    # Transformation des coordonnées pour la méthode buffer
    gdfc["coord_carreau"] = gdfc["coord_carreau"].to_crs(epsg=3491)
    gdfm["geometry"] = gdfm["geometry"].to_crs(epsg=3491)
    gdfs["geometry"] = gdfs["geometry"].to_crs(epsg=3491)

    # Aggrandissement des carreaux des arrondissements et des carreaux 
    polygons = list(gdfm.geometry.buffer(tolerance, cap_style="square")) + list(gdfc.coord_carreau.buffer(tolerance, cap_style="square"))
    
    # Création GeoDataFrame avec les Polygones des arrondissements et les Polygones des carreaux étudiés
    gdf_territoire_etendu = gpd.GeoDataFrame({"geometry" : polygons})
    gdf_territoire_etendu = gdf_territoire_etendu.set_geometry("geometry")

    # Fusion des Polygones : 
    territoire_etendu = gdf_territoire_etendu.union_all()

    # Sélection des arrêts dans la surface étendue
    gdf = gdfs[gdfs.within(territoire_etendu)]

    gdf = gdf.to_crs(epsg=4326)
    territoire_etendu = gpd.GeoSeries({"geometry" : territoire_etendu}, crs = "EPSG:3491").to_crs(epsg=4326)

    return gdf, territoire_etendu
    
    

In [7]:
'''
Certains arrêts sont associés à des lignes dans leur description. Cette fonction vise à les identifier et à les y associer, créant deux
sous-ensembles d'arrêts : un dans lequel une imputation est à faire et l'autre dans lequel elle n'est pas nécessaire.
'''

def col_stop_desc(gdfstop) : 
        
    gdfstop["route_short_name"] = None
    gdfstop_non_imput = gpd.GeoDataFrame(columns = gdfstop.columns)

    cas_speciaux = {"ligne 533 et 582" : ["533", "582"],
                    "départ ligne 36/36B" : ["36", "36B"],
                    "départ/terminus ligne 121/122" : ["121", "122"],
                    "arrêt de dépose 15/15S/16/16S/17" : ["15", "15S", "16", "17"],   # La ligne 16S n'existe plus
                    "arrêt 15 15s 40" : ["15", "15S", "40"],
                    "terminus 17" : ["17"],
                    "terminus 15 15s" : ["15", "15S"],
                    "terminus 24 24b 24t" : ["24"],   # Les lignes 24B et 24T n'existent plus
                    "Prolongement T3" : ["T3"],
                    "M1" : ["M1"]}
    
    for i in gdfstop.index :
        # Description associée à l'arrêt
        desc_i = gdfstop.loc[i, "stop_desc"]
        # Si la valeur n'est pas vide
        if desc_i != None : 
            # S'il y a un code postal, remplacer par une valeur vide
            if re.search(r"13\d{3}", desc_i) :
                gdfstop.loc[i, "stop_desc"] = None
            else : 
                # S'il n'y a pas de nombre, remplacer par une valeur vide
                if not re.search(r"\d+", desc_i): 
                    gdfstop.loc[i, "stop_desc"] = None
                else : 
                    # Cas spéciaux
                    if desc_i in cas_speciaux : 
                        gdf = gdfstop.loc[[i]*len(cas_speciaux[desc_i])].assign(**{'route_short_name': cas_speciaux[desc_i]})
                        gdfstop_non_imput = pd.concat([gdfstop_non_imput, gdf], axis = 0)
                    else : 
                        # S'il y a "ligne" suivi d'un nombre
                        match = re.search(r"ligne \d+", desc_i)
                        if match : 
                            num = re.findall(r"\d+",match.group(0))
                            if num == "82" :    # la ligne 82 n'existe plus
                                gdfstop.loc[i, "stop_desc"] = None
                                continue
                            gdf = gdfstop.loc[[i]*1].assign(**{'route_short_name': num})
                            gdfstop_non_imput = pd.concat([gdfstop_non_imput, gdf], axis = 0)
                        else :
                            gdfstop.loc[i, "stop_desc"] = None

    # Distinction des deux fichiers
    # Suppression des arrêts ayant une ligne 
    gdfstop = gdfstop[gdfstop["stop_desc"].isna()]

    gdfstop = gdfstop.drop(columns = ["stop_desc", "route_short_name"])

    return gdfstop, gdfstop_non_imput

In [8]:
'''
Les données concernant les arrêts ne contiennent quasiment pas d'informations sur les lignes qui y passent. La fonction suivante permet d'associer 
les arrêts aux lignes passant à proximité. Cependant, rien n'indique qu'elles s'y arretent nécessairement.
'''

def buffer_ligne_stop(gdfligne, gdfstop) :

    # Nécessaire pour la boucle ci-dessous
    join = pd.DataFrame({"index_right" : [None] * 11})

    gdfl = gdfligne.copy()
    gdfs = gdfstop.copy()

    # Transformation des coordonnées pour la méthode buffer
    gdfl["geometry"] = gdfl["geometry"].to_crs(epsg=3491)
    gdfs["geometry"] = gdfs["geometry"].to_crs(epsg=3491)
    
    # Tant que le nombre d'arrêt sans ligne attribué est supérieur à 10
    while len(join[join["index_right"].isna()]) > 10 :

        # Augmente la taille de la zone des lignes
        gdfl["buffer"] = gdfl.buffer(25, cap_style = "square")

        gdfl = gdfl.set_geometry("buffer")
        #gdfligne = gdfligne.to_crs(epsg = 3491)

        # Jointure de façon à ce que chaque arrêt dans la zone d'une ligne soit associé à celle-ci
        join = gdfs.sjoin(gdfl, predicate = "within", how = "left")

    join = join.to_crs(epsg=4326)
    join = join.drop(columns = "geometry_right")

    return join

In [9]:
'''
L'obectif de cette fonction est de créer des "zones d'arrêt". En effet, puisque les arrêts ne sont pas assignés à des lignes précises, il a fallu 
créer la fonction buffer_ligne_stop. Mais si les arrêts sont proches, ils peuvent se voir assigner aux mêmes lignes sans que cela soit effectivement 
le cas dans la réalité. Aussi, il peut être plus sur de créer une zone dans laquelle sont contenus ces arrêts et à laquelle seront assignés les lignes
de transports en commun.
dict_na est un dictionnaire reliant le nom d'un arrêt non assigné à la ou les lignes qui doivent lui être.
'''

def zones_arret(gdf_stop, gdf_ligne, dict_na) :

    rows = []

    # Pour chaque nom d'arrêt unique
    for stop_name, gdf_i in gdf_stop.groupby("stop_name"):

        # Extraction du nom des lignes passant par la zone
        lignes = gdf_i["route_short_name"].unique()

        # Nombre d'arrêt
        nb_arrets = len(gdf_i.index.unique())

        # Eviter les doublons dans le nom des lignes
        if stop_name in dict_na.keys() :
            lignes[pd.isnull(lignes)] = dict_na[stop_name]
            lignes = list(set(lignes))

        # Nombre de lignes
        nb_lignes = len(lignes) 
        
        if len(lignes) == 1 :
            lignes = lignes[0]

        # Extraction type des lignes
        typ = gdf_i["route_type"].unique()

        if stop_name in dict_na.keys() :
            typ[pd.isnull(typ)] = gdf_ligne[gdf_ligne["route_short_name"] == dict_na[stop_name]]["route_type"].values
            typ = list(set(typ))
            
        if len(typ) == 1 : 
            typ = typ[0]

        # On ajoute le booléen bus 
        bool_bus = ("Bus" in typ)

         # On ajoute le booléen tram 
        bool_tram = ("Tram" in typ)

        # On ajoute le booléen metro 
        bool_metro = ("Subway" in typ)

        # On ajoute le booléen ferry 
        bool_ferry = ("Ferry" in typ)

        # On crée la geometry
        if nb_arrets == 1 :
            geom = gdf_i["geometry"].values.unique()[0]
        else :
            geom = MultiPoint(gdf_i["geometry"].values.unique())

        rows.append(
            {
                "nom_zone" : stop_name,    # Nom de la zone tiré du nom des arrêts
                "nombre_arret" : nb_arrets,   
                "nom_ligne" : lignes,
                "nombre_ligne" : nb_lignes,
                "type_ligne" : typ, 
                "bool_bus" : bool_bus,
                "bool_tram" : bool_tram,
                "bool_metro" : bool_metro,
                "bool_ferry" : bool_ferry,
                "geometry" : geom
            }
        )

    gdf = gpd.GeoDataFrame(rows, geometry = "geometry", crs = "EPSG:4326")
    gdf["geometry"] = gdf.concave_hull()

    return gdf

In [10]:
'''
Le but de cette fonction est d'associer des zones d'arrêts aux carreaux. 
'''
def carac_carreaux(gdfcarreaux, gdfzone, vieux_port):

    motif = re.compile(r"\w+")

    gdfc = gdfcarreaux.to_crs(3491).copy()
    gdfz = gdfzone.to_crs(3491).copy()

    # nettoyage 
    gdfz["nom_ligne"] = (
        gdfz["nom_ligne"]
        .map(lambda x: x.decode("utf-8") if isinstance(x, bytes) else x)
    )

    # dataframe résultat
    gdfci = gdfc.copy()

    # Distances minimales à une zone d'arrêts avec i lignes
    colonnes_dist = []

    for i in range(1, 7):

        suffixe = "" if i == 1 else "s"
        col_dist = f"dist_min_{i}_ligne{suffixe}"

        gdfzi = gdfz[gdfz["nombre_ligne"] >= i]

        tmp = gdfc.sjoin_nearest(
            gdfzi,
            how="left",
            distance_col=col_dist
        )[[col_dist]]

        gdfci[col_dist] = tmp.groupby(level=0)[col_dist].first()

        colonnes_dist.append(col_dist)

    # Traitement des rayons
    rayons = {
        0: "inside",
        100: "100m",
        200: "200m"
    }

    for dist, suffixe in rayons.items():

        sj = gdfc.sjoin(
            gdfz,
            how="left",
            predicate="dwithin",
            distance=dist
        )

        # nombre de zones
        gdfci[f"zones_{suffixe}"] = (
            sj["nom_zone"]
            .notna()
            .groupby(level=0)
            .sum()
            .reindex(gdfci.index, fill_value=0)
            .astype(int)
        )

        # extraction des lignes
        lignes = (
            sj["nom_ligne"]
            .dropna()
            .astype(str)
            .str.findall(motif)
            .groupby(level=0)
            .sum()
            .map(lambda x: list(dict.fromkeys(x)))
        )

        lignes = lignes.reindex(gdfci.index)

        gdfci[f"nom_lignes_{suffixe}"] = lignes.map(
            lambda x: x if isinstance(x, list) else []
        )

        gdfci[f"nombre_lignes_{suffixe}"] = (
            gdfci[f"nom_lignes_{suffixe}"]
            .str.len()
        )


    # Distance au Vieux-Port
    transformer = Transformer.from_crs(
        4326,
        3491,
        always_xy=True
    )
    
    x, y = transformer.transform(vieux_port[1], vieux_port[0])
    vieux_port_point = Point(x, y)
    gdfci["dist_vieux_port"] = gdfci.geometry.distance(vieux_port_point)

    return gdfci.to_crs(4326)

---
## 4. Observations et transformations des bases de données
---

### 4.1 Données carroyées

In [11]:
# Sélection des carreaux qui se trouvent au moins en partie sur le territoire marseillais
df_carreaux_clean = df_carreaux_clean[df_carreaux_clean["lcog_geo"].str.contains("|".join(code_geo_marseille))]

In [12]:
df_carreaux_clean.head()

,idcar_200m,idcar_1km,idcar_nat,i_est_200,i_est_1km,lcog_geo,ind,men,men_pauv,men_1ind,...,ind_4_5,ind_6_10,ind_11_17,ind_18_24,ind_25_39,ind_40_54,ind_55_64,ind_65_79,ind_80p,ind_inc
44082,CRS3035RES200mN2244800E3948400,CRS3035RES1000mN2244000E3948000,CRS3035RES4000mN2244000E3948000,1,1,13209,1,0.6,0.2,0.4,...,0,0.1,0,0,0.1,0.3,0.1,0.3,0.1,0
44324,CRS3035RES200mN2245000E3948200,CRS3035RES1000mN2245000E3948000,CRS3035RES4000mN2244000E3948000,1,1,13209,11,7.1,2,4.3,...,0,0.8,0.4,0,0.8,3.5,0.8,3.9,0.8,0
44325,CRS3035RES200mN2245000E3949800,CRS3035RES1000mN2245000E3949000,CRS3035RES4000mN2244000E3948000,1,1,13209,10,6.4,1.8,3.9,...,0,0.7,0.4,0,0.7,3.2,0.7,3.6,0.7,0
44326,CRS3035RES200mN2245000E3950000,CRS3035RES1000mN2245000E3950000,CRS3035RES4000mN2244000E3948000,1,1,13209,1,0.6,0.2,0.4,...,0,0.1,0,0,0.1,0.3,0.1,0.3,0.1,0
44562,CRS3035RES200mN2245200E3947800,CRS3035RES1000mN2245000E3947000,CRS3035RES4000mN2244000E3944000,1,1,13209,4,1.5,0.1,0.1,...,0.2,0.3,0,0.5,0.5,0.8,0.9,0.7,0,0


In [13]:
df_carreaux_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3243 entries, 44082 to 76118
Data columns (total 34 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   idcar_200m  3243 non-null   object
 1   idcar_1km   3243 non-null   object
 2   idcar_nat   3243 non-null   object
 3   i_est_200   3243 non-null   object
 4   i_est_1km   3243 non-null   object
 5   lcog_geo    3243 non-null   object
 6   ind         3243 non-null   object
 7   men         3243 non-null   object
 8   men_pauv    3243 non-null   object
 9   men_1ind    3243 non-null   object
 10  men_5ind    3243 non-null   object
 11  men_prop    3243 non-null   object
 12  men_fmp     3243 non-null   object
 13  ind_snv     3243 non-null   object
 14  men_surf    3243 non-null   object
 15  men_coll    3243 non-null   object
 16  men_mais    3243 non-null   object
 17  log_av45    3243 non-null   object
 18  log_45_70   3243 non-null   object
 19  log_70_90   3243 non-null   object
 20  log_ap90

Observations : 
---
- Il y a 34 colonnes et 3 243 lignes
- Il n'y a aucune valeur manquante
- Les colonnes "idcar_1km", "idcar_nat", "i_est_1km" doivent être supprimées
- Seule la colonne "idcar_200m" est du bon type
- La colonne "i_est_200" doit être changée en type booléen
- Les autres colonnes doivent être changée en type numérique

In [14]:
# Suppression colonnes inutiles 
df_carreaux_clean = df_carreaux_clean.drop(columns = ["idcar_1km", "idcar_nat", "i_est_1km"])

# Transformation en type numérique 
for i in df_carreaux_clean.columns.drop(["idcar_200m", "i_est_200", "lcog_geo"]) :
    df_carreaux_clean[i] = pd.to_numeric(df_carreaux_clean[i])

# Transformation en type booléen 
df_carreaux_clean["i_est_200"] = df_carreaux_clean["i_est_200"].apply(eval).astype(bool)

### 4.2 Création de nouvelles variables

In [15]:
# Taux de ménages pauvres (capé à 80%)
df_carreaux_clean["tx_men_pauv"] = df_carreaux_clean["men_pauv"] / df_carreaux_clean["men"]

# Taux de ménages propriétaires
df_carreaux_clean["tx_men_prop"] = df_carreaux_clean["men_prop"] / df_carreaux_clean["men"]

# Taux de ménages habitant dans un logement social
df_carreaux_clean["tx_log_soc"] = df_carreaux_clean["log_soc"] / df_carreaux_clean["men"]

# Nombre de mineurs dans le carreau
df_carreaux_clean["mineurs"] = df_carreaux_clean["ind_0_3"]+df_carreaux_clean["ind_4_5"]+df_carreaux_clean["ind_6_10"]+df_carreaux_clean["ind_11_17"]

# Niveau de vie winsorisé moyen des individus (mineurs inclus)
df_carreaux_clean["ndv_moy_tous"] = df_carreaux_clean["ind_snv"] / df_carreaux_clean["ind"]

# Niveau de vie winsorisé moyen des invidius (mineurs exclus)
df_carreaux_clean["ndv_moy_adulte"] = df_carreaux_clean["ind_snv"] / (df_carreaux_clean["ind"]-df_carreaux_clean["mineurs"])

In [16]:
# Création d'un GeoDataFrame avec les coordonnées des carreaux et leur identifiant
gdf_coord_carreaux = code_to_square_coord(df_carreaux_clean["idcar_200m"])

### 4.3 Présentation du territoire étudié

In [17]:
# Surface géographique du territoire de la ville de Marseille
gdf_contour_clean = select_marseille(gdf_contour_clean)

In [18]:
# Tracé du territoire étudié dans ce travail
territoire_etudie = tracer_territoire_etudie(gdf_contour_clean, gdf_coord_carreaux)

a = folium.Map(vieux_port, zoom_start=12)

folium.GeoJson(territoire_etudie).add_to(a)

a

### 4.4 Données sur les lignes de transport en commun

In [19]:
gdf_ligne_clean.head()

,route_id,route_short_name,route_long_name,route_type,route_color,route_url,geo_point_2d,geometry
0,RTM-27,39,Métro Malpassé - Résidence Fondacle,Bus,312783,None,"{'lon': 5.434461061268498, 'lat': 43.321693356...","MULTILINESTRING ((5.41553 43.32043, 5.41543 43..."
1,RTM-91,44,Métro Rond Point du Prado - Collège Roy d'Espagne,Bus,E63323,None,"{'lon': 5.387402618995015, 'lat': 43.255111465...","MULTILINESTRING ((5.39238 43.27109, 5.39193 43..."
2,RTM-6,4B,Métro La Rose - Les 3 Lucs,Bus,009FE3,None,"{'lon': 5.448994359639043, 'lat': 43.322185414...","MULTILINESTRING ((5.42987 43.33318, 5.42973 43..."
3,RTM-9,5,Métro La Rose - La Parade,Bus,009FE3,None,"{'lon': 5.432006437720254, 'lat': 43.348286603...","MULTILINESTRING ((5.42935 43.33354, 5.42931 43..."
4,RTM-18,7B,Foch 5 Avenues - Bois Lemaître,Bus,FBBA00,None,"{'lon': 5.423007963190277, 'lat': 43.307983389...","MULTILINESTRING ((5.39774 43.30201, 5.39804 43..."


In [20]:
gdf_ligne_clean.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 126 entries, 0 to 125
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype   
---  ------            --------------  -----   
 0   route_id          126 non-null    object  
 1   route_short_name  126 non-null    object  
 2   route_long_name   126 non-null    object  
 3   route_type        126 non-null    object  
 4   route_color       126 non-null    object  
 5   route_url         0 non-null      object  
 6   geo_point_2d      126 non-null    object  
 7   geometry          126 non-null    geometry
dtypes: geometry(1), object(7)
memory usage: 8.0+ KB


In [21]:
# Suppression des colonnes inutiles 
gdf_ligne_clean = gdf_ligne_clean.drop(columns = ["route_url", "geo_point_2d"])

# Pour compléter le code couleur qui est en hexadécimal
gdf_ligne_clean["route_color"] = "#" + gdf_ligne_clean["route_color"]

### 4.5 Données sur les arrêts 

In [22]:
# Sélection des arrêts situés dans le territoire étudié ou dans un périmètre de 250m
gdf_arret_clean, territoire_etendu = select_stop(gdf_arret_clean, gdf_coord_carreaux, gdf_contour_clean, extension)

Certains noms d'arrêts entrainent une mauvaise association spatiale au moment où sont créées les zones d'arrêts et doivent donc être changés

In [23]:
# Changement de nom pour l'arrêt RTM-00004339
index_estaque = gdf_arret_clean[gdf_arret_clean["stop_id"] == "RTM-00004339"].index
gdf_arret_clean.loc[index_estaque, "stop_name"] = "L'Estaque-Ferry"

# Changement de nom pour les arrêts RTM-00002388 et RTM-00002275
index_la_rose = gdf_arret_clean[gdf_arret_clean["stop_id"].isin(["RTM-00002388","RTM-00002275"])].index
gdf_arret_clean.loc[index_la_rose, "stop_name"] = "Métro La Rose"

# Changement de nom pour les arrêts RTM-00002277 et RTM-00002292
index_frais_vallon = gdf_arret_clean[gdf_arret_clean["stop_id"].isin(["RTM-00002277", "RTM-00002292"])].index
gdf_arret_clean.loc[index_frais_vallon, "stop_name"] = "Métro Frais Vallon"

# Changement de nom pour les arrêts RTM-00002293 et RTM-00002278
index_malpasse = gdf_arret_clean[gdf_arret_clean["stop_id"].isin(["RTM-00002293", "RTM-00002278"])].index
gdf_arret_clean.loc[index_malpasse, "stop_name"] = "Métro Malpassé"

# Changement de nom pour l'arrêt RTM-00001241 
index_rouguiere = gdf_arret_clean[gdf_arret_clean["stop_id"] == "RTM-00001241"].index
gdf_arret_clean.loc[index_rouguiere, "stop_name"] = "Les Caillols Hôpital"

In [24]:
gdf_arret_clean.head()

,stop_id,stop_code,stop_name,stop_desc,zone_id,stop_url,location_type,parent_station,stop_timezone,wheelchair_boarding,geometry
0,RTM-00000992,00000992,4 Chemins des Aygalades,None,None,None,0,None,None,2,POINT (5.36553 43.34639)
1,RTM-00000212,00000212,5 Avenues Burel,None,None,None,0,None,None,2,POINT (5.39009 43.31757)
2,RTM-00001218,00001218,AFPA La Treille,None,None,None,0,None,None,2,POINT (5.50973 43.30811)
3,RTM-00001970,00001970,Aiguier CNRS,None,None,None,0,None,None,2,POINT (5.40515 43.25623)
4,RTM-00002000,00002000,Aiguier CPCAM,None,None,None,0,None,None,2,POINT (5.41376 43.25082)


In [25]:
gdf_arret_clean.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
Index: 2578 entries, 0 to 2750
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype   
---  ------               --------------  -----   
 0   stop_id              2578 non-null   object  
 1   stop_code            2578 non-null   object  
 2   stop_name            2578 non-null   object  
 3   stop_desc            798 non-null    object  
 4   zone_id              0 non-null      object  
 5   stop_url             0 non-null      object  
 6   location_type        2578 non-null   object  
 7   parent_station       0 non-null      object  
 8   stop_timezone        0 non-null      object  
 9   wheelchair_boarding  2578 non-null   object  
 10  geometry             2578 non-null   geometry
dtypes: geometry(1), object(10)
memory usage: 306.2+ KB


Observations : 
---
- Ce fichier se compose de 11 colonnes et 2578 lignes
- 4 colonnes sont complétement vides : zone_id, stop_url, parent_station, stop_timezone
- La colonne stop_desc comporte des valeurs vides
- Toutes ne sont pas au bon format, mais les seules qui vont être conservées sont les 4 premières et la colonne geometry

In [26]:
# Suppression colonne inutile 
#gdf_arret_clean = gdf_arret_clean.drop(columns = ["zone_id", "stop_url", "location_type", "parent_station", "stop_timezone", "wheelchair_boarding"])

# Séparation en deux sous-ensembles
gdf_arret_imput, gdf_arret_non_imput = col_stop_desc(gdf_arret_clean)

In [27]:
gdf_arret_non_imput.head()

,stop_id,stop_code,stop_name,stop_desc,zone_id,stop_url,location_type,parent_station,stop_timezone,wheelchair_boarding,geometry,route_short_name
30,RTM-00004763,00004763,Aubert Ganay,Prolongement T3,None,None,0,None,None,1,POINT (5.40308 43.26601),T3
321,RTM-00004107,00004107,Hôpital Nord,depart/terminus ligne 97,None,None,0,None,None,1,POINT (5.36344 43.37736),97
322,RTM-00004108,00004108,Hôpital Nord,départ/terminus ligne 121/122,None,None,0,None,None,2,POINT (5.3639 43.37706),121
322,RTM-00004108,00004108,Hôpital Nord,départ/terminus ligne 121/122,None,None,0,None,None,2,POINT (5.3639 43.37706),122
323,RTM-00004767,00004767,Hôpital Sainte-Marguerite,Prolongement T3,None,None,0,None,None,1,POINT (5.40871 43.25941),T3


In [28]:
gdf_arret_non_imput.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
Index: 61 entries, 30 to 2632
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype   
---  ------               --------------  -----   
 0   stop_id              61 non-null     object  
 1   stop_code            61 non-null     object  
 2   stop_name            61 non-null     object  
 3   stop_desc            61 non-null     object  
 4   zone_id              0 non-null      object  
 5   stop_url             0 non-null      object  
 6   location_type        61 non-null     object  
 7   parent_station       0 non-null      object  
 8   stop_timezone        0 non-null      object  
 9   wheelchair_boarding  61 non-null     object  
 10  geometry             61 non-null     geometry
 11  route_short_name     61 non-null     object  
dtypes: geometry(1), object(11)
memory usage: 6.2+ KB


In [29]:
# Imputation des lignes aux arrêts non identifiés
gdf_arret_imput = buffer_ligne_stop(gdf_ligne_clean, gdf_arret_imput)

In [30]:
gdf_arret_imput.head()

,stop_id,stop_code,stop_name,zone_id,stop_url,location_type,parent_station,stop_timezone,wheelchair_boarding,geometry,index_right,route_id,route_short_name,route_long_name,route_type,route_color
0,RTM-00000992,00000992,4 Chemins des Aygalades,None,None,0,None,None,2,POINT (5.36553 43.34639),88.0,RTM-17,27,Métro La Rose - Lycée Saint-Exupéry,Bus,#6AAADE
1,RTM-00000212,00000212,5 Avenues Burel,None,None,0,None,None,2,POINT (5.39009 43.31757),13.0,RTM-20,31,Canebière Bourse - Les Aygalades,Bus,#009640
1,RTM-00000212,00000212,5 Avenues Burel,None,None,0,None,None,2,POINT (5.39009 43.31757),14.0,RTM-22,33,Réformés Canebière - St Jérôme Parking Relais,Bus,#00B1B2
1,RTM-00000212,00000212,5 Avenues Burel,None,None,0,None,None,2,POINT (5.39009 43.31757),32.0,RTM-108,533,Canebière Bourse - Géraniums,Bus,#009640
1,RTM-00000212,00000212,5 Avenues Burel,None,None,0,None,None,2,POINT (5.39009 43.31757),38.0,RTM-10218,N2,Vieux-Port - Einstein Parking Relais,Bus,#005D86


In [31]:
gdf_arret_imput.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
Index: 6986 entries, 0 to 2750
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype   
---  ------               --------------  -----   
 0   stop_id              6986 non-null   object  
 1   stop_code            6986 non-null   object  
 2   stop_name            6986 non-null   object  
 3   zone_id              0 non-null      object  
 4   stop_url             0 non-null      object  
 5   location_type        6986 non-null   object  
 6   parent_station       0 non-null      object  
 7   stop_timezone        0 non-null      object  
 8   wheelchair_boarding  6986 non-null   object  
 9   geometry             6986 non-null   geometry
 10  index_right          6984 non-null   float64 
 11  route_id             6984 non-null   object  
 12  route_short_name     6984 non-null   object  
 13  route_long_name      6984 non-null   object  
 14  route_type           6984 non-null   object  
 15  route_color       

In [32]:
# Jointure arrêts sans imputation et lignes
gdf_arret_non_imput = gdf_arret_non_imput.merge(gdf_ligne_clean.drop(columns = "geometry"), on = "route_short_name", how = "left")

# Concaténation des sous-ensembles d'arrêts
gdf_arret_tout = pd.concat([gdf_arret_non_imput, gdf_arret_imput])

In [33]:
# Problème d'imputation : un exemple 
test = gdf_arret_tout[(gdf_arret_tout["stop_name"].str.contains("Prado")) & (gdf_arret_tout["route_type"] == "Subway")]
lat = test.loc[1618, "geometry"].x
lon = test.loc[1618, "geometry"].y

m = folium.Map([lon,lat], zoom_start=15)

folium.GeoJson(test).add_to(m)

m

L'ensemble des arrêts représentés sur la carte vont être associés à la ligne de métro M2, alors que les arrêts réels de celle-ci ne sont situés qu'aux deux extrémités de cet ensemble.

In [34]:
# Arrêts sans ligne affectée
gdf_arret_imput[gdf_arret_imput["route_short_name"].isna()]

,stop_id,stop_code,stop_name,zone_id,stop_url,location_type,parent_station,stop_timezone,wheelchair_boarding,geometry,index_right,route_id,route_short_name,route_long_name,route_type,route_color
1143,RTM-00002771,00002771,Escale Borély,None,None,0,None,None,2,POINT (5.37695 43.25444),NaN,NaN,NaN,NaN,NaN,NaN
2008,RTM-00002770,00002770,Escale Borély,None,None,0,None,None,2,POINT (5.37679 43.25458),NaN,NaN,NaN,NaN,NaN,NaN


In [35]:
# Repérer les arrets non assignés et les lignes à proximité

lat = gdf_arret_imput[gdf_arret_imput["route_short_name"].isna()].loc[1143,"geometry"].x
lon = gdf_arret_imput[gdf_arret_imput["route_short_name"].isna()].loc[1143,"geometry"].y

m = folium.Map([lon,lat], zoom_start=18)
folium.GeoJson(gdf_ligne_clean,
              popup = folium.GeoJsonPopup(fields=["route_short_name"])).add_to(m) 
folium.GeoJson(gdf_arret_imput[gdf_arret_imput["route_short_name"].isna()],
              popup = folium.GeoJsonPopup(fields=["stop_name", "stop_id"])).add_to(m)

m

Après vérification directement sur le site de la RTM, ces arrêts sont bien relié à la ligne 47.

In [36]:
# Dictionnaire pour lier les arrêts sans ligne assignée avec la bonne ligne
dict_na = {"Escale Borély" : "47"}

### 4.6 Données sur les zones 

In [37]:
# Création des zones en fonction des noms des arrêts et détermination de leurs caractéristiques vis-à-vis des lignes qui y passent 
gdf_zones = zones_arret(gdf_arret_tout, gdf_ligne_clean, dict_na)

In [38]:
gdf_zones.head()

,nom_zone,nombre_arret,nom_ligne,nombre_ligne,type_ligne,bool_bus,bool_tram,bool_metro,bool_ferry,geometry
0,4 Chemins des Aygalades,4,"[27, 30]",2,Bus,True,False,False,False,"POLYGON ((5.36573 43.34623, 5.36553 43.34639, ..."
1,5 Avenues Burel,4,"[31, 33, 533, N2, 32, 34]",6,Bus,True,False,False,False,"POLYGON ((5.3901 43.31713, 5.38955 43.31708, 5..."
2,5 Avenues Chartreux,1,"[81, 88, 42T, 42]",4,Bus,True,False,False,False,POINT (5.39756 43.30328)
3,AFPA La Treille,2,"[S6, 12S]",2,Bus,True,False,False,False,"LINESTRING (5.50973 43.30811, 5.50951 43.30806)"
4,Aiguier CNRS,2,"[48, 47]",2,Bus,True,False,False,False,"LINESTRING (5.40515 43.25623, 5.40534 43.25597)"


In [39]:
gdf_zones.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 1276 entries, 0 to 1275
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype   
---  ------        --------------  -----   
 0   nom_zone      1276 non-null   object  
 1   nombre_arret  1276 non-null   int64   
 2   nom_ligne     1276 non-null   object  
 3   nombre_ligne  1276 non-null   int64   
 4   type_ligne    1276 non-null   object  
 5   bool_bus      1276 non-null   bool    
 6   bool_tram     1276 non-null   bool    
 7   bool_metro    1276 non-null   bool    
 8   bool_ferry    1276 non-null   bool    
 9   geometry      1276 non-null   geometry
dtypes: bool(4), geometry(1), int64(2), object(3)
memory usage: 64.9+ KB


In [40]:
# Proportion des zones par rapport au total des arrêts 
round(len(gdf_zones)/len(gdf_arret_clean), 2)

0.49

Observations :
---
- On retrouve 1276 zones avec un nom différent
- Le passage aux zones d'arrêts a diminué de 50% le nombre d'entrées
- Il n'y a aucune valeur manquante
- Les colonnes sont du bon type

In [41]:
# Visualisation de la surface étudiée et des zones d'arrêts
m = folium.Map(vieux_port, zoom_start=12, tiles="cartodbpositron")

folium.GeoJson(territoire_etendu).add_to(m)
folium.GeoJson(gdf_zones[["geometry", "nom_zone", "nombre_arret"]], 
              popup = folium.GeoJsonPopup(fields=["nom_zone", "nombre_arret"])).add_to(m) 

m

### 4.7 Association entre les zones d'arrêts et les données carroyées

In [42]:
# Association des carreaux aux zones en leur sein, à 100m et à 200m, ainsi qu'aux zones les plus proches en fonction du nombre de lignes qui y passent
gdf_coord_carreaux_infos = carac_carreaux(gdf_coord_carreaux, gdf_zones, vieux_port)

# Jointure entre les deux fichiers de données carroyées
gdf_carreaux = gdf_coord_carreaux_infos.merge(df_carreaux_clean, on = "idcar_200m")

---
## 5. Base de données finale
---

In [43]:
gdf_carreaux.head()

,idcar_200m,coord_carreau,dist_min_1_ligne,dist_min_2_lignes,dist_min_3_lignes,dist_min_4_lignes,dist_min_5_lignes,dist_min_6_lignes,zones_inside,nom_lignes_inside,...,ind_55_64,ind_65_79,ind_80p,ind_inc,tx_men_pauv,tx_men_prop,tx_log_soc,mineurs,ndv_moy_tous,ndv_moy_adulte
0,CRS3035RES200mN2244800E3948400,"POLYGON ((5.42447 43.20883, 5.42692 43.20894, ...",1855.744620,2454.888686,2454.888686,2454.888686,5195.633470,5726.409914,0,[],...,0.1,0.3,0.1,0.0,0.333333,0.333333,0.0,0.1,26675.100000,29639.000000
1,CRS3035RES200mN2245000E3948200,"POLYGON ((5.42188 43.21052, 5.42433 43.21063, ...",1611.116700,2366.624934,2366.624934,2366.624934,4971.509498,5476.243246,0,[],...,0.8,3.9,0.8,0.0,0.281690,0.281690,0.0,1.2,26675.109091,29941.448980
2,CRS3035RES200mN2245000E3949800,"POLYGON ((5.44149 43.2114, 5.44394 43.21151, 5...",1987.039237,1987.039237,1987.039237,2050.572528,5383.297073,6118.871157,0,[],...,0.7,3.6,0.7,0.0,0.281250,0.281250,0.0,1.1,26675.110000,29972.033708
3,CRS3035RES200mN2245000E3950000,"POLYGON ((5.44394 43.21151, 5.44639 43.21162, ...",1982.034035,1982.034035,1982.034035,2079.635536,5465.632003,6221.913302,0,[],...,0.1,0.3,0.1,0.0,0.333333,0.333333,0.0,0.1,26675.100000,29639.000000
4,CRS3035RES200mN2245200E3947800,"POLYGON ((5.41683 43.21211, 5.41929 43.21222, ...",1374.148418,2236.820209,2236.820209,2435.782860,4743.870592,5182.195762,0,[],...,0.9,0.7,0.0,0.0,0.066667,0.066667,0.0,0.6,31369.675000,36905.500000


In [44]:
gdf_carreaux.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 3243 entries, 0 to 3242
Data columns (total 54 columns):
 #   Column                Non-Null Count  Dtype   
---  ------                --------------  -----   
 0   idcar_200m            3243 non-null   object  
 1   coord_carreau         3243 non-null   geometry
 2   dist_min_1_ligne      3243 non-null   float64 
 3   dist_min_2_lignes     3243 non-null   float64 
 4   dist_min_3_lignes     3243 non-null   float64 
 5   dist_min_4_lignes     3243 non-null   float64 
 6   dist_min_5_lignes     3243 non-null   float64 
 7   dist_min_6_lignes     3243 non-null   float64 
 8   zones_inside          3243 non-null   int64   
 9   nom_lignes_inside     3243 non-null   object  
 10  nombre_lignes_inside  3243 non-null   int64   
 11  zones_100m            3243 non-null   int64   
 12  nom_lignes_100m       3243 non-null   object  
 13  nombre_lignes_100m    3243 non-null   int64   
 14  zones_200m            3243 non-null   int64   
 

Observations : 
---
- On retrouve les 3243 lignes du fichiers de départ
- On a désormais 54 colonnes
- Il n'y a aucune valeur manquante
- Les types sont les bons 

Synthèse de la construction de la base de donnée : 
---
- Au départ 4 bases différentes
- Plusieurs combinaisons ont été réalisées :
  - Détermination de la surface géographique étudiée
  - Sélection des arrêts de transport en commun dans cette surface ou à proximité
  - Association des arrêts à des lignes de transport
  - Création de zones regroupant plusieurs arrêts du même nom
  - Utilisation de ces zones pour enrichir les données carroyées INSEE
- Au final, une base de données sur des carreaux de 200mx200m regroupant des informations socio-démographiques compilées par l'INSEE et des informations sur l'accessibilité au réseau de transports en commun
- Cette base a pour but d'être exportée et mobilisée dans un second notebook "02_analysis" afin d'étudier les relations entre ces informations.

---
## 6. Export des données
---

In [45]:
gdf_carreaux.to_file("../data/cleaned/data_marseille_tec_cleaned.geojson")
gdf_zones.to_file("../data/cleaned/data_zones_arrets.geojson")